# llm-serving-benchmark: Kaggle runner

Runs the full benchmark on a Kaggle **T4** GPU:
HF `generate()` baseline vs vLLM 0.18.0 under concurrency.

**Before running:** Settings -> Accelerator: **GPU T4**, Internet: **on**. Set `REPO_URL` below.
Then *Run All*. Results are zipped to `/kaggle/working/results.zip` at the end.

In [ ]:
REPO_URL = "https://github.com/karimmike/llm-serving-benchmark.git" 
MODELS = ["qwen-fp16", "qwen-awq"]  # keys in configs/experiments.yaml
HF_BASELINE_MODEL = "qwen-fp16"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # Kaggle may give 2x T4; benchmark exactly one

In [ ]:
!nvidia-smi

## 1. Clone the repo and install pinned requirements

In [ ]:
WORKDIR = "/kaggle/working/llm-serving-benchmark"
!git clone -q {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!pip install -q -r requirements-kaggle.txt
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__)"

## 2. Helpers: start/stop the vLLM server and run commands with live output

The server command comes from `configs/experiments.yaml`, so the notebook and the config can't drift apart.

In [ ]:
import os, signal, subprocess, sys, time, urllib.request
from pathlib import Path

sys.path.insert(0, WORKDIR)
from bench.config import load_config, server_url, vllm_serve_command

cfg = load_config()
LOG_DIR = Path(WORKDIR) / "results" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)


def run(cmd):
    """Run a command, streaming its output into the notebook; fail loudly."""
    print("$", " ".join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}")


def start_server(model_key):
    cmd = vllm_serve_command(cfg, model_key)
    log_path = LOG_DIR / f"vllm_{model_key}.log"
    print("$", " ".join(cmd), ">", log_path)
    log = open(log_path, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    return proc, log_path


def wait_for_health(proc, log_path, timeout_s=900):
    url = f"{server_url(cfg)}/health"
    start = time.time()
    while time.time() - start < timeout_s:
        if proc.poll() is not None:
            print(log_path.read_text()[-5000:])
            raise RuntimeError(f"vLLM exited early (code {proc.returncode}); log tail above")
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    print(f"vLLM healthy after {time.time() - start:.0f}s")
                    return
        except OSError:
            pass
        time.sleep(5)
    print(log_path.read_text()[-5000:])
    raise TimeoutError(f"vLLM not healthy after {timeout_s}s")


def stop_server(proc):
    os.killpg(proc.pid, signal.SIGTERM)  
        proc.wait(timeout=60)
    except subprocess.TimeoutExpired:
        os.killpg(proc.pid, signal.SIGKILL)
        proc.wait()
    time.sleep(5)  # GPU memory released

## 3. vLLM sweep: every model x workload x concurrency level

In [ ]:
for model_key in MODELS:
    proc, log_path = start_server(model_key)
    try:
        wait_for_health(proc, log_path)
        run([sys.executable, "scripts/run_benchmark.py", "--engine", "vllm", "--model", model_key])
    finally:
        stop_server(proc)

## 4. HF `generate()` baseline (server is stopped, so it has the GPU to itself)

In [ ]:
!nvidia-smi --query-gpu=memory.used --format=csv
run([sys.executable, "scripts/run_benchmark.py", "--engine", "hf", "--model", HF_BASELINE_MODEL])

## 5. Plots and download

In [ ]:
run([sys.executable, "scripts/plot_results.py"])

from IPython.display import FileLink, Image, display
for png in sorted((Path(WORKDIR) / "results" / "plots").glob("*.png")):
    display(Image(filename=str(png)))

In [ ]:
import shutil
archive = shutil.make_archive("/kaggle/working/results", "zip", root_dir=WORKDIR, base_dir="results")
print(archive)
%cd /kaggle/working
FileLink("results.zip")  